<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-11/AI_Modul_11.1_Praktikum_Pengenalan_OpenCV.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 11.1: Praktikum Pengenalan OpenCV
**Mata Kuliah:** Visi Komputer dan Pengolahan Citra Digital  
**Institut Pertanian Stiper (INSTIPER) Yogyakarta**  

---

### Capaian Pembelajaran Praktikum:
1. Memeriksa versi, kapabilitas build, dan akselerasi perangkat keras OpenCV (OpenCL, thread CPU).
2. Memahami representasi matriks citra NumPy dan urutan kanal BGR vs RGB.
3. Menguji perbedaan operasi aritmatika saturasi OpenCV (`cv2.add`) vs aritmatika modulo NumPy.
4. Melakukan tolok ukur (*benchmarking*) kecepatan eksekusi manipulasi citra Full HD pada data perkebunan kelapa sawit.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import time

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print(f'OpenCV Version : {cv2.__version__}')
print(f'NumPy Version  : {np.__version__}')
print(f'Jumlah Thread  : {cv2.getNumThreads()}')
print(f'Dukungan OpenCL: {cv2.ocl.haveOpenCL()}')


## 1. Pembangkitan Citra Sintetis Kanopi Sawit dan Struktur Kanal BGR
Membangkitkan citra sintetis kanopi kelapa sawit dan menganalisis susunan matriks 3 dimensi (Tinggi, Lebar, Kanal BGR).

In [ ]:
height, width = 400, 600
canopy_img = np.zeros((height, width, 3), dtype=np.uint8)

Y, X = np.ogrid[:height, :width]
cx, cy = width // 2, height // 2
radius = np.sqrt((X - cx)**2 + (Y - cy)**2)

# Kanal 0: Biru (Blue) - rendah pada daun sehat
canopy_img[:, :, 0] = np.clip(50 - radius * 0.1, 10, 60).astype(np.uint8)
# Kanal 1: Hijau (Green) - tinggi pada daun sehat
canopy_img[:, :, 1] = np.clip(200 - radius * 0.35, 40, 220).astype(np.uint8)
# Kanal 2: Merah (Red) - sedang pada kanopi sawit
canopy_img[:, :, 2] = np.clip(80 - radius * 0.15, 20, 100).astype(np.uint8)

print(f'Dimensi Matriks Citra (H, W, C) : {canopy_img.shape}')
print(f'Tipe Data Matriks               : {canopy_img.dtype}')
print(f'Nilai Piksel Pusat Kanopi (BGR) : {canopy_img[cy, cx]}')


## 2. Aritmatika Saturasi OpenCV (cv2.add) vs Modulo Overflow NumPy
Mendemonstrasikan mengapa operasi penjumlahan piksel wajib menggunakan `cv2.add` untuk mencegah *overflow artifact*.

In [ ]:
val1 = np.uint8([220])
val2 = np.uint8([50])

# Penjumlahan NumPy biasa (Modulo 256 Overflow)
res_numpy = val1 + val2

# Penjumlahan OpenCV (Saturated Arithmetic)
res_opencv = cv2.add(val1, val2)

print(f'Nilai Awal: {val1[0]} + {val2[0]}')
print(f'Hasil NumPy Biasa (Modulo 256) : {res_numpy[0]}  <-- ARTIFAK CACAT!')
print(f'Hasil OpenCV (Saturated)       : {res_opencv[0][0]} <-- BENAR (Maksimal 255)')


## 3. Tolok Ukur Kecepatan Pemrosesan Citra Resolusi Tinggi
Membandingkan efisiensi waktu komputasi pemrosesan Full HD (1920x1080) antara fungsi teroptimasi OpenCV C++ dan operasi NumPy.

In [ ]:
hd_frame = np.random.randint(0, 256, (1080, 1920, 3), dtype=np.uint8)

# 1. Uji cv2.add (SIMD C++)
t0 = time.perf_counter()
for _ in range(50):
    res1 = cv2.add(hd_frame, np.array([25, 25, 25, 0], dtype=np.uint8)[:3])
t_cv2 = (time.perf_counter() - t0) / 50.0

# 2. Uji NumPy Vectorized
t0 = time.perf_counter()
for _ in range(50):
    res2 = np.clip(hd_frame.astype(np.int16) + 25, 0, 255).astype(np.uint8)
t_np = (time.perf_counter() - t0) / 50.0

print(f'Rata-rata Waktu cv2.add : {t_cv2 * 1000.0:.3f} ms')
print(f'Rata-rata Waktu NumPy   : {t_np * 1000.0:.3f} ms')
print(f'Rasio Kecepatan OpenCV  : {t_np / max(t_cv2, 1e-6):.2f}x lebih mangkus')

# Visualisasi Hasil Pengujian dalam bentuk Bar Chart
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['cv2.add (C++ SIMD)', 'NumPy Vectorized'], [t_cv2 * 1000, t_np * 1000], color=['#1565C0', '#E65100'])
ax.set_ylabel('Latensi per Frame (ms)', fontsize=10)
ax.set_title('Komparasi Latensi Pemrosesan Citra Full HD', fontweight='bold', fontsize=11)
plt.tight_layout()
plt.savefig('praktikum_benchmark_opencv_11_1.png', dpi=150)
plt.show()
